# Credit Card Default Analysis & Visualization using Python

**Goal:** understand which customer characteristics and repayment behaviours are linked to credit card payment default, using exploratory data analysis (EDA).

**Dataset:** *Default of Credit Card Clients* (UCI Machine Learning Repository, also on Kaggle) - 30,000 customers, 23 features and a target column showing whether the customer defaulted on the next month's payment.

**Libraries:** Python, NumPy, Pandas, Matplotlib, Seaborn

**Steps:** 1. Load data, 2. Clean data, 3. Overall default rate, 4. Demographics, 5. Credit limit and age, 6. Repayment history, 7. Bill and payment behaviour, 8. Correlations, 9. Key insights

In [ ]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 110

IMG_DIR = Path("images")
IMG_DIR.mkdir(exist_ok=True)

def save(name):
    plt.tight_layout()
    plt.savefig(IMG_DIR / f"{name}.png", bbox_inches="tight")
    plt.show()

## 1. Load the data
Download the dataset (see README) and place it at `data/UCI_Credit_Card.csv`.

In [ ]:
DATA_PATH = Path("data/UCI_Credit_Card.csv")
df = pd.read_csv(DATA_PATH)

# Standardise column names (Kaggle / UCI versions differ slightly)
df = df.rename(columns={
    "PAY_0": "PAY_1",
    "default.payment.next.month": "DEFAULT",
    "default payment next month": "DEFAULT",
})
print("Shape:", df.shape)
df.head()

In [ ]:
df.info()
df.describe().T

## 2. Data cleaning
Checks: missing values, duplicates, and category codes that are not in the dataset documentation
(`EDUCATION` should be 1-4, `MARRIAGE` should be 1-3).

In [ ]:
print("Missing values:", int(df.isna().sum().sum()))
print("Duplicate customer IDs:", int(df["ID"].duplicated().sum()))
print("Duplicate rows (ignoring ID):", int(df.drop(columns="ID").duplicated().sum()))

print("\nEDUCATION codes:", sorted(df["EDUCATION"].unique()))
print("MARRIAGE codes:", sorted(df["MARRIAGE"].unique()))

In [ ]:
# Remove duplicate rows (ignoring ID), then drop ID
before = len(df)
df = df.loc[~df.drop(columns="ID").duplicated()].drop(columns="ID").reset_index(drop=True)
print(f"Removed {before - len(df)} duplicate rows")

# Group undocumented category codes
df["EDUCATION"] = df["EDUCATION"].replace({0: 4, 5: 4, 6: 4})   # 4 = others
df["MARRIAGE"] = df["MARRIAGE"].replace({0: 3})                  # 3 = others

# Readable labels for plotting
df["SEX_LABEL"] = df["SEX"].map({1: "Male", 2: "Female"})
df["EDU_LABEL"] = df["EDUCATION"].map({1: "Graduate school", 2: "University", 3: "High school", 4: "Others"})
df["MARRIAGE_LABEL"] = df["MARRIAGE"].map({1: "Married", 2: "Single", 3: "Others"})

# Feature engineering
df["AGE_GROUP"] = pd.cut(df["AGE"], bins=[20, 30, 40, 50, 60, 100],
                         labels=["21-30", "31-40", "41-50", "51-60", "60+"])
df["LIMIT_BAND"] = pd.qcut(df["LIMIT_BAL"], 4, labels=["Low", "Medium", "High", "Very high"])
df["UTILIZATION"] = df["BILL_AMT1"] / df["LIMIT_BAL"]

df.shape

## 3. Overall default rate

In [ ]:
default_rate = df["DEFAULT"].mean() * 100
print(f"Overall default rate: {default_rate:.2f}%")

fig, ax = plt.subplots(figsize=(5, 4))
sns.countplot(data=df, x="DEFAULT", ax=ax)
ax.set_xticklabels(["No default", "Default"])
ax.set_xlabel("")
ax.set_title(f"Customers by default status (default rate {default_rate:.1f}%)")
save("01_default_distribution")

## 4. Demographics vs default rate

In [ ]:
def rate_plot(ax, col, title, order=None):
    rates = df.groupby(col)["DEFAULT"].mean().mul(100)
    if order is not None:
        rates = rates.reindex(order)
    sns.barplot(x=rates.index.astype(str), y=rates.values, ax=ax)
    ax.axhline(default_rate, color="red", ls="--", lw=1, label="Overall")
    ax.set_title(title)
    ax.set_xlabel("")
    ax.set_ylabel("Default rate (%)")
    ax.legend()

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
rate_plot(axes[0], "SEX_LABEL", "By gender")
rate_plot(axes[1], "EDU_LABEL", "By education",
          ["Graduate school", "University", "High school", "Others"])
rate_plot(axes[2], "MARRIAGE_LABEL", "By marital status")
save("02_demographics_default_rate")

## 5. Credit limit and age

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

sns.histplot(data=df, x="LIMIT_BAL", hue="DEFAULT", bins=40, multiple="stack", ax=axes[0])
axes[0].set_title("Credit limit distribution")

rate_plot(axes[1], "LIMIT_BAND", "Default rate by credit limit band")
rate_plot(axes[2], "AGE_GROUP", "Default rate by age group")
save("03_limit_and_age")

## 6. Repayment history
`PAY_1` is the repayment status of the most recent month: -2 = no consumption, -1 = paid in full,
0 = revolving credit / minimum payment, 1 to 8 = months of payment delay.

In [ ]:
pay_counts = df["PAY_1"].value_counts()
pay_rates = df.groupby("PAY_1")["DEFAULT"].mean().mul(100)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
sns.barplot(x=pay_counts.sort_index().index, y=pay_counts.sort_index().values, ax=axes[0])
axes[0].set_title("Customers by latest repayment status (PAY_1)")
axes[0].set_ylabel("Customers")

sns.barplot(x=pay_rates.index, y=pay_rates.values, ax=axes[1])
axes[1].axhline(default_rate, color="red", ls="--", lw=1, label="Overall")
axes[1].set_title("Default rate by latest repayment status")
axes[1].set_ylabel("Default rate (%)")
axes[1].legend()
save("04_repayment_status")

In [ ]:
pay_cols = [f"PAY_{i}" for i in range(1, 7)]
df["MONTHS_DELAYED"] = (df[pay_cols] > 0).sum(axis=1)

delay_rates = df.groupby("MONTHS_DELAYED")["DEFAULT"].mean().mul(100)
fig, ax = plt.subplots(figsize=(7, 4))
sns.barplot(x=delay_rates.index, y=delay_rates.values, ax=ax)
ax.set_xlabel("Number of delayed months (out of last 6)")
ax.set_ylabel("Default rate (%)")
ax.set_title("More delayed months, higher default rate")
save("05_months_delayed")

## 7. Bill and payment behaviour

In [ ]:
df["UTIL_BAND"] = pd.cut(df["UTILIZATION"], bins=[-np.inf, 0, 0.25, 0.5, 0.75, 1, np.inf],
                         labels=["<=0", "0-25%", "25-50%", "50-75%", "75-100%", ">100%"])

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
rate_plot(axes[0], "UTIL_BAND", "Default rate by credit utilization (latest bill / limit)")

months = ["Apr", "May", "Jun", "Jul", "Aug", "Sep"]
avg_bill = [df[f"BILL_AMT{i}"].mean() for i in range(6, 0, -1)]
avg_pay = [df[f"PAY_AMT{i}"].mean() for i in range(6, 0, -1)]
axes[1].plot(months, avg_bill, marker="o", label="Average bill amount")
axes[1].plot(months, avg_pay, marker="o", label="Average payment amount")
axes[1].set_title("Average bill vs payment amount (6 months)")
axes[1].set_ylabel("NT dollars")
axes[1].legend()
save("06_utilization_and_bills")

## 8. Correlation analysis

In [ ]:
num_cols = ["LIMIT_BAL", "AGE", "PAY_1", "PAY_2", "PAY_3", "BILL_AMT1", "PAY_AMT1", "PAY_AMT2",
            "UTILIZATION", "MONTHS_DELAYED", "DEFAULT"]
corr = df[num_cols].corr()

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=ax)
ax.set_title("Correlation heatmap")
save("07_correlation_heatmap")

corr["DEFAULT"].drop("DEFAULT").sort_values(key=abs, ascending=False)

## 9. Key insights (generated from the data)

In [ ]:
def top_rate(col, min_n=100):
    g = df.groupby(col)["DEFAULT"].agg(["mean", "count"])
    g = g[g["count"] >= min_n]
    return g["mean"].idxmax(), g["mean"].max() * 100, g["mean"].idxmin(), g["mean"].min() * 100

edu_hi, edu_hi_r, _, _ = top_rate("EDU_LABEL")
age_hi, age_hi_r, age_lo, age_lo_r = top_rate("AGE_GROUP")
lim_hi, lim_hi_r, lim_lo, lim_lo_r = top_rate("LIMIT_BAND")
pay_hi, pay_hi_r, pay_lo, pay_lo_r = top_rate("PAY_1")
util_hi, util_hi_r, _, _ = top_rate("UTIL_BAND")
strongest = corr["DEFAULT"].drop("DEFAULT").abs().idxmax()

print(f"1. Overall default rate is {default_rate:.1f}% of {len(df):,} customers.")
print(f"2. Latest repayment status (PAY_1): highest default rate is {pay_hi_r:.1f}% for PAY_1 = {pay_hi}, "
      f"lowest is {pay_lo_r:.1f}% for PAY_1 = {pay_lo}.")
print(f"3. Strongest correlation with default: {strongest} ({corr.loc[strongest, 'DEFAULT']:.2f}).")
print(f"4. Credit limit: '{lim_hi}' band defaults at {lim_hi_r:.1f}% vs {lim_lo_r:.1f}% for '{lim_lo}'.")
print(f"5. Age: {age_hi} defaults highest ({age_hi_r:.1f}%), {age_lo} lowest ({age_lo_r:.1f}%).")
print(f"6. Education: '{edu_hi}' has the highest default rate ({edu_hi_r:.1f}%).")
print(f"7. Utilization: '{util_hi}' band has the highest default rate ({util_hi_r:.1f}%).")

## Conclusion
- Recent repayment behaviour (`PAY_1`, number of delayed months) separates defaulters from non-defaulters far better than demographics.
- Demographic factors (gender, education, marital status, age) show smaller differences and should be used with care.
- For a collections team, customers with delayed payments and high credit utilization are the natural early-warning group to prioritise.

**Next steps:** build a predictive model (for example logistic regression or random forest) on these features and compare its performance.